# 1 · Dataset Validation

Audits the raw Roboflow export and the leakage-repaired split.

Run `python scripts/validate_dataset.py` first - this notebook reads its saved artefacts rather than recomputing them, so what you see here is exactly what the report cites.

In [ ]:
import sys, json
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))   # import the project's src/ package
import pandas as pd
from IPython.display import Image, display
from src import paths


## Raw dataset audit

In [ ]:
raw = json.loads((paths.VALIDATION_OUTPUT_DIR / 'raw' / 'validation_report.json').read_text())
print('classes:', raw['classes'])
print('images per class:', raw['images_per_class_total'])
print('>=200 images per class:', raw['min_200_images_per_class'])
pd.DataFrame(raw['splits']).T[['images','label_files','background_images','images_with_both_classes','label_issue_count']]

## Leakage: what we found, and the repair

Mirrored/noise-augmented copies and sequential frames of the same source image were scattered across train/valid/test. Groups - not images - were then reassigned to splits.

In [ ]:
rs = json.loads((paths.VALIDATION_OUTPUT_DIR / 'resplit_report.json').read_text())
print(f"images                : {rs['images']:,}")
print(f"source groups         : {rs['source_groups']:,}")
print(f"BEFORE - groups spanning splits: {rs['leakage_before']['groups_spanning_splits']}")
print(f"BEFORE - images in those groups: {rs['leakage_before']['images_in_spanning_groups']:,}")
print(f"AFTER  - groups spanning splits: {rs['leakage_after']['groups_spanning_splits']}")
pd.DataFrame(rs['new_split_counts']).T

## Audit of the repaired split

In [ ]:
proc = json.loads((paths.VALIDATION_OUTPUT_DIR / 'processed' / 'validation_report.json').read_text())
print('cross-split exact duplicates:', proc['exact_duplicates_cross_split'])
print('images per class:', proc['images_per_class_total'])
pd.DataFrame(proc['splits']).T[['images','background_images','images_with_both_classes']]